<img src="../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

<img src="assets/agent08-top-light.svg" alt="Four reaction paths, each a chain of images between its own two ends, sit in one batch; one NEB call relaxes all four onto their lowest paths, and the top image of each path is marked." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 5px solid #76b900; border-radius: 4px; padding: 24px 28px; margin: 8px 0 18px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="font-variant: small-caps; font-size: 12px; font-weight: 700; letter-spacing: 2px; opacity: 0.75;">AGENTIC TRACK, NOTEBOOK 08</div><h1 style="margin: 6px 0 8px; padding: 0; border: 0;"><span style="font-size: 32px; font-weight: 700; line-height: 1.2; color: inherit;">Paths of least resistance, batched</span></h1><div style="font-size: 18px; line-height: 1.45; opacity: 0.85;">Find reaction paths for many systems in one batch and test their saddle points.</div></div>

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 12px 18px; margin: 12px 0 18px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; line-height: 1.6;"><b>Hardware:</b> a CUDA-capable NVIDIA GPU<br><b>Prerequisites:</b> Meet your agent; Made to order<br><b>References:</b> <a href="https://nvidia.github.io/nvalchemi-toolkit/" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Toolkit documentation</a>; <a href="https://nvidia.github.io/nvalchemi-toolkit/examples/" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Toolkit examples</a>; <a href="https://github.com/NVIDIA/skills/tree/main/skills/nvalchemi-toolkit" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">the nvalchemi-toolkit skill</a></div>

<img src="assets/agent08-roadmap-light.svg" alt="Sections of this notebook on one line." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">

A reaction's rate is set by its barrier: the highest point on the lowest
path from reactant to product. That point is a saddle point. The nudged
elastic band (NEB) method finds the path as a chain of images between the
two ends. Toolkit keeps the chains of many reactions in one batch and
relaxes them in one call. You will build four paths, relax them together,
and test what sits at the top of each.

In [ ]:
# Load the course helpers, the model, four reactions and the NEB settings
import torch
from helpers import agent08 as helpers
helpers.configure_tutorial()             # course display defaults
work_dir = helpers.make_work_dir()       # notebooks/agent08_work
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = helpers.load_model(device)
reactants, products, reaction_names = helpers.load_reactions(model)
neb_settings = dict(num_images=8, fmax=0.003, regular_fmax=0.03, n_steps=600)  # images per path, both ends; eV/Å; eV/Å; steps
helpers.show_session()                   # prints the session lines

`model` is AIMNet2 with its Coulomb term. Each reaction comes as two
relaxed ends, ASE `Atoms` with the same atom order:

In [ ]:
# Show the two ends of each reaction
helpers.show_reactions(reactants, products, reaction_names)

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">1. Every path in one batch</h2>

A path starts as a straight line of images between its two ends. Toolkit
draws that line for every reaction at once and returns one `Batch`, in
which each path is a group of images.

#### Example: build the paths

| | |
| --- | --- |
| input  | `reactants`; `products`; `neb_settings` |
| output | `paths`, in `agent08_work/paths.py` |

> 💡 **Hint**
>
> `helpers.show_paths(...)` draws the result. `paths` has energy, forces and
> velocities attached: `interpolate_paths` drops them, and the NEB run reads
> all three.

In [ ]:
%load agent08_work/paths.py

In [ ]:
# Show every path and its images
helpers.show_paths(paths, reaction_names)

Each row is one path. Every image of every path sits in the same `Batch`.

> </> API
>
> `interpolate_paths(initial, final, num_images=...)`
>
> **Input:** two batches, the reactants and the products<br>
> **Effect:** one `Batch` with one group of images per path

> 💡 **Hint**
>
> Atoms must be in the same order at both ends, and the two ends must lie
> over each other. The loader has done both here. For a reaction of your
> own, ask your agent to check them first.

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">2. One call for every path</h2>

One `NEB` call relaxes every path in the batch. With a climbing image, the
highest image of each path is pushed uphill along the path until it sits
at the top. Each path stops on its own, when its forces are small enough.
Ask for the run's diagnostics file as well: it records every path at
every step.

A path that stopped is not always converged: it also stops when its step
budget runs out. A barrier counts only if the path's last force in the
log is below the threshold of the run.

#### Try it: relax the paths and read the barriers

Relax every path with one `NEB` call and a climbing image, with
`neb_settings`. Then read the barrier of each path and whether the path
converged.

| | |
| --- | --- |
| input  | `paths`; `model`; `neb_settings` |
| output | `relaxed_paths`, `neb_log`, `path_energy`, `barriers`, `converged`, in `agent08_work/neb.py` |

> 💡 **Hint**
>
> `helpers.plot_energy_profiles(...)` draws the result. `barriers` is one
> number per path, in eV: the forward barrier, from the reactant end to the
> top of the path.

Done when: the table under the plot gives a barrier and a converged yes or
no for every path.

In [ ]:
%load agent08_work/neb.py

In [ ]:
# Show where each path stopped
helpers.show_neb_log(neb_log, reaction_names)

Each path stopped at its own step, and the call ended with the slowest
one.

> ⚙️ **Fine control**
>
> ```python
> NEB(
>     model=...,
>     fmax=...,       # eV/Å, where each path stops
>     n_steps=...,    # step budget of the whole run
>     climbing=ClimbingImageConfig(
>         mode="after_regular",
>         regular_fmax=...,   # eV/Å, where climbing starts
>     ),
> )
> ```
>
> Example: `run again and start climbing at 0.05 eV/Å`

In [ ]:
# Plot the energy along every path
helpers.plot_energy_profiles(relaxed_paths, path_energy, reaction_names, barriers, converged)

Each curve runs from reactant to product. Its highest point is the
barrier.

> </> API
>
> `NEB(..., diagnostics_log_path=...)`
>
> **Input:** a file path<br>
> **Effect:** a CSV with one row per path at each logged step: `fmax`, `energy_barrier`, `path_length`; `energy_barrier` counts from the lower end

> </> API
>
> `batch.group_layout`
>
> **Holds:** the groups of a `Batch`<br>
> **Effect:** here each path is one group of images

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">3. The top of each path</h2>

The top image of a converged path should be a saddle point: a structure
with no net force on any atom, where the energy falls along exactly one
direction and rises along every other.

#### Try it: saddle point or not

Find a way in Toolkit to tell whether a structure is a saddle point. Test
every image of your paths.

| | |
| --- | --- |
| input  | `relaxed_paths`; `model` |
| output | `saddle_table`, in `agent08_work/saddle.py` |

> 💡 **Hint**
>
> `helpers.show_saddle_table(...)` draws the result. `saddle_table` has one
> column each for the path (the names in `reaction_names`), the image number,
> the largest force on an atom, the count of imaginary modes and the yes or
> no. A yes needs a largest force at or below 0.005 eV/Å. Project the rigid
> translations and rotations out of the Hessian, then count the imaginary
> modes.

Done when: the table gives a yes or no for every image, with the numbers
behind it.

In [ ]:
%load agent08_work/saddle.py

In [ ]:
# Show your agent's verdict for every image
helpers.show_saddle_table(saddle_table)

Read one path from end to end: its two ends, the images on the slope, the
top.

> 💡 **Hint**
>
> A transition state has forces near zero and exactly one imaginary
> frequency. Check both: images near the top of a path can show one
> imaginary mode too.

> </> API
>
> `model.compute_hessian(batch)`
>
> **Input:** a `Batch` that carries its neighbor data<br>
> **Effect:** the dense Hessian is written to `batch["hessian"]`

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">Explore more</h2>

More ways to find and test reaction paths with Toolkit:

<div style="display: flex; flex-wrap: wrap; gap: 16px; margin: 4px 0 18px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div style="overflow-wrap: break-word;"><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/advanced/11_batched_neb.py">A better first path</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8; overflow-wrap: break-word;">IDPP refines the straight line before the model is called.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div style="overflow-wrap: break-word;"><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/advanced/11_batched_neb.py">Other optimizers and force rules</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8; overflow-wrap: break-word;"><code>LBFGS</code> in <code>NEB</code>, or your own NEB force equation.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div style="overflow-wrap: break-word;"><b><a href="https://nvidia.github.io/nvalchemi-toolkit/userguide/models.html#hessians-and-hessian-vector-products">Hessians without the matrix</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8; overflow-wrap: break-word;">Hessian-vector products for systems too large for a dense Hessian.</div></div></div>

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; padding: 15px 20px; margin: 24px 0 18px; line-height: 1.6;"><h2 style="margin: 0 0 10px; padding: 0; border: 0; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 24px; font-weight: 500; line-height: 1.25; color: inherit;">Notebook 08 recap</h2><div style="margin: 0 0 10px;">You built four reaction paths in one <code>Batch</code>, relaxed them with one <code>NEB</code> call, and tested the top of each for a saddle point.</div><div style="margin: 0;"><img src="assets/fork-agent-light.svg" alt="" width="18" height="18" style="width: 18px; height: 18px; vertical-align: -3px; margin-right: 6px;"><b>Next:</b> <a href="alchemi-agent-09-small-batch-distilled.ipynb" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Small-batch distilled →</a></div></div>